<a href="https://colab.research.google.com/github/26b21a4254kiet-create/PHEONIX/blob/main/IMAGE_TRANSFORMATION_DECOMPOSER_(back_end)_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# IMAGE TRANSFORMATION DECOMPOSER (B10)
# B.Tech Mathematics Mini Project
# Eigenvalues, Eigenvectors & Geometric Interpretation
# Google Colab - Single Cell Version
# ============================================================

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

np.set_printoptions(precision=4, suppress=True)

# ============================================================
# 1. USER CONFIGURATION
# ============================================================
# Change these four numbers to test any 2x2 transformation matrix.

A = np.array([
    [2.0, 1.0],
    [1.0, 2.0]
])

# Example benchmark matrix:
# A = [[2, 1],
#      [1, 2]]
#
# Its eigenvalues are 3 and 1.
# Its eigenvectors are along [1,1] and [1,-1].

# Example image/object used for visualization
square = np.array([
    [-1, -1],
    [ 1, -1],
    [ 1,  1],
    [-1,  1]
], dtype=float)

# ============================================================
# 2. INPUT VALIDATION
# ============================================================

if A.shape != (2, 2):
    raise ValueError("The transformation matrix must be 2 × 2.")

print("=" * 70)
print("        IMAGE TRANSFORMATION DECOMPOSER (B10)")
print("=" * 70)

print("\nINPUT TRANSFORMATION MATRIX A:")
print(A)

# ============================================================
# 3. MATHEMATICAL MODEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: MATHEMATICAL MODEL")
print("=" * 70)

print("""
For a transformation matrix A, an eigenvector v satisfies:

              A v = λ v

where:
    A  = transformation matrix
    v  = eigenvector
    λ  = corresponding eigenvalue

The eigenvalue tells us how much the vector is stretched
(or compressed) in its eigenvector direction.
""")

# ============================================================
# 4. EIGENVALUES USING SYMPY
# ============================================================

print("=" * 70)
print("STEP 2: FINDING EIGENVALUES")
print("=" * 70)

A_sym = sp.Matrix(A)

# Characteristic polynomial
lambda_symbol = sp.symbols('λ')
characteristic_polynomial = A_sym.charpoly(lambda_symbol).as_expr()

print("\nCharacteristic polynomial:")
print("det(A - λI) =", characteristic_polynomial)

eigenvalues_sym = A_sym.eigenvals()

print("\nEigenvalues:")
for value, multiplicity in eigenvalues_sym.items():
    print(f"λ = {value}   (multiplicity = {multiplicity})")

# Numerical eigenvalues
eigenvalues, eigenvectors_numpy = np.linalg.eig(A)

print("\nNumerical eigenvalues:")
for i, value in enumerate(eigenvalues):
    print(f"λ{i+1} = {value:.6f}")

# ============================================================
# 5. EIGENVECTORS
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: FINDING EIGENVECTORS")
print("=" * 70)

eigenvectors_sym = A_sym.eigenvects()

real_eigenvectors = []

for eigenvalue, multiplicity, vectors in eigenvectors_sym:

    print(f"\nEigenvalue λ = {eigenvalue}")

    for v in vectors:
        print("Eigenvector:")
        sp.pprint(v)

        # Check whether eigenvalue/eigenvector is real
        if all(element.is_real for element in v) and eigenvalue.is_real:
            v_np = np.array(v, dtype=float).reshape(2)

            # Normalize vector
            norm = np.linalg.norm(v_np)

            if norm != 0:
                v_np = v_np / norm

            real_eigenvectors.append(
                (float(eigenvalue), v_np)
            )

# ============================================================
# 6. VERIFY EIGENVALUE-EIGENVECTOR RELATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: VERIFYING A v = λ v")
print("=" * 70)

if len(real_eigenvectors) == 0:

    print("""
No real eigenvectors were found.

This means the transformation does not have real eigendirections.
The transformation may still be perfectly valid, but its
geometric behavior cannot be explained using real eigenvector
directions.
""")

else:

    for i, (lam, v) in enumerate(real_eigenvectors):

        Av = A @ v
        lambda_v = lam * v

        print(f"\nEigenvector {i+1}")
        print("v =", v)
        print("A v =", Av)
        print("λ v =", lambda_v)

        if np.allclose(Av, lambda_v):
            print("✓ Verified: A v = λ v")
        else:
            print("✗ Verification failed")

# ============================================================
# 7. GEOMETRIC INTERPRETATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: GEOMETRIC INTERPRETATION")
print("=" * 70)

if len(real_eigenvectors) == 0:

    print("""
There are no real eigenvector directions for this matrix.

Therefore we cannot demonstrate pure stretching along real
eigendirections.
""")

else:

    for i, (lam, v) in enumerate(real_eigenvectors):

        print(f"\nDirection {i+1}:")
        print("Eigenvector =", v)
        print("Eigenvalue  =", lam)

        if abs(lam) > 1:
            print("→ Stretching occurs.")
        elif abs(lam) < 1:
            print("→ Compression occurs.")
        elif abs(lam) == 1:
            print("→ Length is preserved.")
        else:
            print("→ Vector is mapped to zero.")

        if lam < 0:
            print("→ Negative eigenvalue also reverses direction.")

# ============================================================
# 8. SINGULAR VALUE / STRETCHING ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("STEP 6: ROTATION + STRETCHING ANALYSIS")
print("=" * 70)

print("""
For a completely general matrix, the mathematically reliable
rotation/stretch decomposition is the Polar Decomposition:

              A = R S

where:
    R = orthogonal part (rotation/reflection)
    S = symmetric positive-semidefinite stretching part.

This is different from saying that every matrix has stretching
only along its eigenvectors.

We calculate the Polar Decomposition using SVD.
""")

# SVD:
# A = U Σ V^T
U, singular_values, VT = np.linalg.svd(A)

# Orthogonal part
R = U @ VT

# Symmetric stretching part
S = VT.T @ np.diag(singular_values) @ VT

print("\nSingular values (principal stretch factors):")
for i, s in enumerate(singular_values):
    print(f"s{i+1} = {s:.6f}")

print("\nRotation / orthogonal part R:")
print(R)

print("\nStretching part S:")
print(S)

print("\nVerification of A ≈ R @ S:")
print(R @ S)

if np.allclose(A, R @ S):
    print("✓ Polar decomposition verified.")
else:
    print("✗ Decomposition verification failed.")

# Determine whether R represents a proper rotation
det_R = np.linalg.det(R)

print(f"\ndet(R) = {det_R:.6f}")

if det_R > 0:
    rotation_angle = np.arctan2(R[1, 0], R[0, 0])
    rotation_degrees = np.degrees(rotation_angle)

    print(
        f"Rotation angle ≈ {rotation_degrees:.4f} degrees"
    )

else:
    print("""
The orthogonal part contains a reflection rather than a pure
rotation because det(R) < 0.
""")

# ============================================================
# 9. TRANSFORM OBJECT
# ============================================================

print("\n" + "=" * 70)
print("STEP 7: TRANSFORMING AN OBJECT")
print("=" * 70)

transformed_square = square @ A.T

print("Original points:")
print(square)

print("\nTransformed points:")
print(transformed_square)

# ============================================================
# 10. PLOT

        IMAGE TRANSFORMATION DECOMPOSER (B10)

INPUT TRANSFORMATION MATRIX A:
[[2. 1.]
 [1. 2.]]

STEP 1: MATHEMATICAL MODEL

For a transformation matrix A, an eigenvector v satisfies:

              A v = λ v

where:
    A  = transformation matrix
    v  = eigenvector
    λ  = corresponding eigenvalue

The eigenvalue tells us how much the vector is stretched
(or compressed) in its eigenvector direction.

STEP 2: FINDING EIGENVALUES

Characteristic polynomial:
det(A - λI) = 1.0*λ**2 - 4.0*λ + 3.0

Eigenvalues:
λ = 1.00000000000000   (multiplicity = 1)
λ = 3.00000000000000   (multiplicity = 1)

Numerical eigenvalues:
λ1 = 3.000000
λ2 = 1.000000

STEP 3: FINDING EIGENVECTORS

Eigenvalue λ = 1.00000000000000
Eigenvector:
⎡-0.707106781186548⎤
⎢                  ⎥
⎣0.707106781186548 ⎦

Eigenvalue λ = 3.00000000000000
Eigenvector:
⎡-0.707106781186548⎤
⎢                  ⎥
⎣-0.707106781186548⎦

STEP 4: VERIFYING A v = λ v

Eigenvector 1
v = [-0.7071  0.7071]
A v = [-0.7071  0.7071]
λ v = [-0.